## Import thư viện cần thiết

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

## Get dữ liệu từ database load lên dataFrame

In [6]:
database_path = "D:/Code/project_code/Project/data/database/courses_database.db"
try: 
    with sqlite3.connect(database_path) as connection:
        statement = """
                SELECT 
                    c.course_id,
                    c.course_title,
                    c.price,
                    c.level,
                    c.subject,
                    c.source_platform,
                    e.enrollment_id,
                    e.student_name,
                    e.student_email,
                    e.rating_score,
                    e.completion_rate,
                    e.enrollment_date,
                    e.payment_method
                FROM Courses c
                LEFT JOIN Enrollments e ON c.course_id = e.course_id
            """
        df = pd.read_sql_query(statement, connection)
    print("load dữ liệu thành công lên dataFrame")
except sqlite3.Error as e:
    print(e)

load dữ liệu thành công lên dataFrame


In [7]:
df

,course_id,course_title,price,level,subject,source_platform,enrollment_id,student_name,student_email,rating_score,completion_rate,enrollment_date,payment_method
0,1070968,Ultimate Investment Banking Course,200.0,All Levels,Business Finance,Udemy Courses,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1113822,Complete GST Course & Certification - Grow You...,75.0,All Levels,Business Finance,Udemy Courses,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1006314,Financial Modeling for Business Analysts and C...,45.0,Intermediate Level,Business Finance,Udemy Courses,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1210588,Beginner to Pro - Financial Analysis in Excel ...,95.0,All Levels,Business Finance,Udemy Courses,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1011058,How To Maximize Your Profits Trading Options,200.0,Intermediate Level,Business Finance,Udemy Courses,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
12022,ONLINE166972,NaN,NaN,NaN,NaN,Online Courses,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12023,ONLINE914188,NaN,NaN,NaN,NaN,Online Courses,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12024,ONLINE654524,NaN,NaN,NaN,NaN,Online Courses,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12025,ONLINE136897,NaN,NaN,NaN,NaN,Online Courses,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Hiển thị thông tin tổng quan của dataFrame

In [11]:
print(f"kích thước của dữ liệu là: {df.shape}")
print("-"*50)
print("các cột dữ liệu: ")
print(df.columns.tolist())
print("-"*50)
print("kiểu dữ liệu của từng cột là: ")
print(df.dtypes)

kích thước của dữ liệu là: (12027, 13)
--------------------------------------------------
các cột dữ liệu: 
['course_id', 'course_title', 'price', 'level', 'subject', 'source_platform', 'enrollment_id', 'student_name', 'student_email', 'rating_score', 'completion_rate', 'enrollment_date', 'payment_method']
--------------------------------------------------
kiểu dữ liệu của từng cột là: 
course_id              str
course_title           str
price              float64
level                  str
subject                str
source_platform        str
enrollment_id          str
student_name           str
student_email          str
rating_score       float64
completion_rate    float64
enrollment_date        str
payment_method         str
dtype: object


## Clean Data

### 1.Missing Values

#### 1.1 xem thông tin các cột bị thiếu

In [12]:
missing_values = df.isna().sum()
print(missing_values)

course_id              0
course_title        3249
price               8027
level               7911
subject             5273
source_platform        0
enrollment_id      10527
student_name       10527
student_email      10527
rating_score       10527
completion_rate    10527
enrollment_date    10527
payment_method     10527
dtype: int64


#### 1.2 Xử lý các cột bị thiếu

In [13]:
# Điền giá trị mặc định cho tiêu đề và phân loại bị thiếu
df["course_title"] = df['course_title'].fillna('Unknown Course')
df['level'] = df['level'].fillna("Unspecified")
df['subject'] = df['subject'].fillna("Unspecified")
#Điền giá trị bị thiếu của "price" bằng median 
median_price = df['price'].median()
df['price'] = df['price'].fillna(median_price)
# Ta tạo thêm một cột "has_enrollment" để xem trạng thái của khóa học đã có lượt đăng ký chưa:
# has_enrollment = 1 (Đã có học viên đăng ký)
# has_enrollment = 0 (Khóa học chưa có ai đăng ký / ế)
df['has_enrollment'] = df['enrollment_id'].notnull().astype(int)


#### 1.3 kiểm tra lại sau khi xử lý missing values

In [17]:
print("số lượng missing values sau khi xử lý: ")
print(df.isna().sum())
print("-"*50)
print(f"tổng số lượng khóa học trong hệ thống: {len(df)}")
print("-"*50)
print(f"Số lượng khóa học đã được bán ít nhất 1 lần: {df['has_enrollment'].sum()}")
print("-"*50)
print(f"số lượng khóa học chưa có ai đăng ký(ế): {len(df) - df['has_enrollment'].sum()}")

số lượng missing values sau khi xử lý: 
course_id              0
course_title           0
price                  0
level                  0
subject                0
source_platform        0
enrollment_id      10527
student_name       10527
student_email      10527
rating_score       10527
completion_rate    10527
enrollment_date    10527
payment_method     10527
has_enrollment         0
dtype: int64
--------------------------------------------------
tổng số lượng khóa học trong hệ thống: 12027
--------------------------------------------------
Số lượng khóa học đã được bán ít nhất 1 lần: 1500
--------------------------------------------------
số lượng khóa học chưa có ai đăng ký(ế): 10527


### 2. Xử lý duplicate values

#### 1.1 xem thông tin giá trị bị trùng lặp

In [ ]:
duplicate_values = df.duplicated().sum()
print(f"số lượng giá trị bị trùng lắp trong dữ liệu là: {duplicate_values}")


số lượng giá trị bị trùng lắp trong dữ liệu là: 0


## Export dữ liệu đã được làm sạch

In [20]:
cleaned_file_path = "D:/Code/project_code/Project/data/processed/master_dataset_cleaned.csv"
df.to_csv(cleaned_file_path, index=False)
print(f"đã lưu thành công tập dữ liệu làm sạch tại đường dẫn: {cleaned_file_path}")
print(f"kích thước dữ liệu sau khi đã làm sạch là: {df.shape}")

đã lưu thành công tập dữ liệu làm sạch tại đường dẫn: D:/Code/project_code/Project/data/processed/master_dataset_cleaned.csv
kích thước dữ liệu sau khi đã làm sạch là: (12027, 14)
